# Microsoft Planetary Computer Met Office check

In [1]:
import duckdb

con = duckdb.connect()
con.sql("DESCRIBE SELECT * FROM 'output/checks.parquet'")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ collection         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ item               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ reference_datetime │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ has_item           │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ num_missing        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [2]:
con.sql("DESCRIBE SELECT * FROM 'output/paths.parquet'")

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ model              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ reference_datetime │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ path               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ collection         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ item               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

In [ ]:
monthly = con.sql("""
    SELECT
        collection,
        date_trunc('month', reference_datetime) AS month,
        SUM(CASE WHEN NOT has_item THEN 1 ELSE 0 END) AS missing,
        SUM(CASE WHEN has_item AND num_missing > 0 THEN 1 ELSE 0 END) AS incomplete,
        SUM(CASE WHEN has_item AND num_missing = 0 THEN 1 ELSE 0 END) AS complete
    FROM 'output/checks.parquet'
    GROUP BY collection, month
    ORDER BY collection, month
""").df()
monthly

In [ ]:
import matplotlib.pyplot as plt

collections = sorted(monthly["collection"].unique())
fig, axes = plt.subplots(len(collections), 1, figsize=(12, 4 * len(collections)), sharex=True)

for ax, collection in zip(axes, collections):
    data = monthly[monthly["collection"] == collection].set_index("month")
    ax.stackplot(
        data.index,
        data["complete"],
        data["incomplete"],
        data["missing"],
        labels=["complete", "incomplete", "missing"],
        colors=["#2ca02c", "#ff7f0e", "#d62728"],
    )
    ax.set_ylabel("STAC items")
    ax.set_title(collection)
    ax.legend(loc="upper left")

axes[-1].set_xlabel("month")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()